# 04 — Figure Generation

Publication-quality figures: track + intensity, and intensity vs translation speed.
Run notebooks 01–03 first so `../output/` holds the required inputs.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
from src.visualization import apply_publication_style, save_figure
from src.track_utils import haversine_km, translation_speed_kmh

apply_publication_style()
out = pathlib.Path('../output'); out.mkdir(exist_ok=True)
track = pd.read_csv('../data/biparjoy_besttrack_sample.csv', parse_dates=['time_utc'])
track = track.sort_values('time_utc').reset_index(drop=True)


In [ ]:
# Figure 1 — track coloured by intensity
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(track.lon, track.lat, c=track.wind_kt, cmap='viridis', s=40)
ax.plot(track.lon, track.lat, color='0.6', lw=0.8, zorder=0)
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Cyclone Biparjoy track (June 2023)')
plt.colorbar(sc, ax=ax, label='Max sustained wind (kt)')
save_figure(fig, out / 'fig1_track_intensity.png')
plt.show()


In [ ]:
# Figure 2 — intensity vs translation speed
rows = []
for i in range(1, len(track)):
    a, b = track.iloc[i - 1], track.iloc[i]
    d = haversine_km(a.lon, a.lat, b.lon, b.lat)
    dt = (b.time_utc - a.time_utc).total_seconds() / 3600
    rows.append({'wind_kt': b.wind_kt, 'speed_kmh': translation_speed_kmh(d, dt) or 0})
kin = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(kin.speed_kmh, kin.wind_kt)
ax.set_xlabel('Translation speed (km h⁻¹)')
ax.set_ylabel('Max sustained wind (kt)')
ax.set_title('Intensity vs translation speed')
save_figure(fig, out / 'fig2_intensity_vs_speed.png')
plt.show()
